In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
CODE, URL = '/content/code', 'github.com/quocthangtrann/computer_vision_ADAS.git'
env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
if not os.path.exists(CODE):
    subprocess.run(f'git clone https://{URL} {CODE}', shell=True, check=True, env=env)
else:
    subprocess.run(f'git -C {CODE} pull', shell=True, check=True, env=env)
subprocess.run(f'rsync -a --exclude .git {CODE}/repo/ /content/drive/MyDrive/capstone_ai/repo/', shell=True, check=True)
try: import segmentation_models_pytorch
except ImportError: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'segmentation-models-pytorch'], check=True)
os.chdir('/content/drive/MyDrive/capstone_ai')
print('mã mới nhất đã ở capstone_ai/repo/ | thư mục làm việc:', os.getcwd())


In [ ]:
import os, subprocess, sys
os.makedirs('runs', exist_ok=True)
# CẤU HÌNH. A (mặc định): FSNet-M 320x160, run b9s4_c4aug_s0.  B (HD): W, H, ARCH = 640, 320, 'fsnet_m_p1s5', RUN = 'runs/hd640s1_p1s5_kdT_c4aug_s0'
W, H, ARCH = 320, 160, 'fsnet_m'
RUN = 'runs/b9s4_c4aug_s0'      # last.pt["model"] là trọng số triển khai (bản EMA nếu run bật EMA); trùng với tệp trong models_final/
OUT = 'handoff_s3b' if (W, H) == (320, 160) else f'handoff_s3b_{ARCH}_{W}x{H}'   # gói HD ghi ra thư mục khác, không đè gói 320x160
SZ = ['--arch', ARCH, '--w', str(W), '--h', str(H)]

def run(script, *args):
    cmd = [sys.executable, '-u', f'repo/{script}', *args]
    print('>>>', ' '.join(cmd), flush=True)
    pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    with open(f'runs/{script[:-3]}.log', 'a') as lg:
        for ln in pr.stdout:
            lg.write(ln); print(ln, end='', flush=True)
    if pr.wait() != 0: raise SystemExit(f'{script} dừng (mã thoát {pr.returncode}): dán đoạn log trên cho tôi, KHÔNG chạy bước sau')

# Chạy được trên runtime CPU. Bước FP32 của int8_sim so ONNX với final_metrics.json của run: lệch quá 0,002 thì dừng.
if (W, H) != (320, 160): run('make_hd_calib.py', '--w', str(W), '--h', str(H))   # 200 ảnh hiệu chuẩn + 100 ảnh test HD; có rồi thì bỏ qua
run('int8_sim.py', '--run', RUN, '--out', OUT, *SZ)
run('int8_pack.py', '--run', RUN, '--out', OUT, *SZ)
